# Rotors

**Type:** Consumer — pulls from storage, stores output in dimensional depot  
**Blueprint:** `rotor` (60 Iron Rods/min + 300 Screws/min → 12 Rotors/min at 100%)  
**Pulls from:** Iron Rods storage, Screws storage

> Set supply rates below to `[producer output] − [stash rate]`:
> - Iron Rods: check `iron-rods.ipynb` → use `available_for_rotors`
> - Screws: check `screws.ipynb` → use a portion of `available_for_downstream`  
>   (the rest goes to Reinforced Iron Plates)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from blueprints import BLUEPRINTS, Blueprint, Stage
from satisfactory import ITEMS
import pandas as pd

BP  = BLUEPRINTS
TOL = 0.5


def machine_hover(bp: Blueprint, output_key: str, target_rate: float) -> str:
    if not bp.stages:
        return ""
    rates = bp.stage_rates(output_key, target_rate)
    lines = ["── Machines (per copy) ──"]
    for r in rates:
        name = ITEMS[r["item_key"]].name
        lines.append(f"  {r['machines']}× {r['building'].title()} → {r['per_machine_rate']:.2f} {name}/min each")
    return "<br>" + "<br>".join(lines)

## Constants

In [ ]:
# ── Supply from storage ───────────────────────────────────────────────────────
# These are the rates this chain PULLS from the dimensional depot each minute.
# Must be less than what the producer notebooks output (leaving the stash rate accumulating).
IRON_RODS_FROM_STORAGE = None  # ← Iron Rods/min pulled from storage
SCREWS_FROM_STORAGE    = None  # ← Screws/min pulled from storage

# ── Blueprint placement ───────────────────────────────────────────────────────
ROTOR_COPIES      = None  # ← number of blueprints to place
ROTOR_OUTPUT_EACH = None  # ← output rate per copy to set in-game (Rotors/min)

## Derived rates and balance

In [ ]:
assert None not in (IRON_RODS_FROM_STORAGE, SCREWS_FROM_STORAGE, ROTOR_COPIES, ROTOR_OUTPUT_EACH), \
    "Fill in all constants in the cell above before running this cell."

rotor_total      = ROTOR_COPIES * ROTOR_OUTPUT_EACH
rods_consumed    = rotor_total  * BP['rotor'].input_ratio('iron-rod', 'rotor')
screws_consumed  = rotor_total  * BP['rotor'].input_ratio('screw',    'rotor')

assert abs(rods_consumed - IRON_RODS_FROM_STORAGE) < TOL, (
    f"Iron Rod balance: consuming {rods_consumed:.2f}/min, available {IRON_RODS_FROM_STORAGE:.2f}/min "
    f"(delta {rods_consumed - IRON_RODS_FROM_STORAGE:+.2f})"
)
assert abs(screws_consumed - SCREWS_FROM_STORAGE) < TOL, (
    f"Screw balance: consuming {screws_consumed:.2f}/min, available {SCREWS_FROM_STORAGE:.2f}/min "
    f"(delta {screws_consumed - SCREWS_FROM_STORAGE:+.2f})"
)

print(f"✓ Chain balanced")
print(f"  {IRON_RODS_FROM_STORAGE:.2f} Iron Rods/min  +  {SCREWS_FROM_STORAGE:.2f} Screws/min")
print(f"  →  {rotor_total:.2f} Rotors/min")
print(f"  Set each of {ROTOR_COPIES} blueprint{'s' if ROTOR_COPIES > 1 else ''} to: {ROTOR_OUTPUT_EACH:.2f} Rotors/min")